In [18]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import Ridge

from utils import read_data


In [ ]:
adata = read_data()

#Ensure target_gene is categorical so we have stable ordering
if not pd.api.types.is_categorical_dtype(adata.obs["target_gene"]):
    adata.obs["target_gene"] = adata.obs["target_gene"].astype("category")

adata


/var/folders/zh/h1c40rms7bzccg0_9q2lrf1c0000gn/T/ipykernel_12839/1961222690.py:4: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs["target_gene"]):


AnnData object with n_obs × n_vars = 221273 × 18080 backed at '/Users/mk/Library/CloudStorage/OneDrive-Personal/VCC/vcc-git/data/vcc_data/adata_Training.h5ad'
    obs: 'target_gene', 'guide_id', 'batch'
    var: 'gene_id'

In [ ]:
#Compute control mean and per-target signatures
ctrl = adata[adata.obs["target_gene"] == "non-targeting"]
mu_ctrl = np.asarray(ctrl.X.mean(axis=0)).ravel()

target_genes = [g for g in adata.obs["target_gene"].cat.categories if g != "non-targeting"]
signatures = []
for g in target_genes:
    Xg = np.asarray(adata[adata.obs["target_gene"] == g].X.mean(axis=0)).ravel()
    signatures.append(Xg - mu_ctrl)

Y = np.stack(signatures)  # shape: (n_targets, n_genes)
X = np.eye(len(target_genes), dtype=float)

print(f"Prepared signatures for {len(target_genes)} targets across {Y.shape[1]} genes.")


Prepared signatures for 150 targets across 18080 genes.


In [ ]:
#Fit a multi-output Ridge model mapping target identity -> expression shift
alpha = 1.0
model = Ridge(alpha=alpha)
model.fit(X, Y)

print("Fit multi-output Ridge model")
print(f"coef shape: {model.coef_.shape}, intercept shape: {model.intercept_.shape}")


Fit multi-output Ridge model
coef shape: (18080, 150), intercept shape: (18080,)


In [ ]:
#Quick reconstruction error on the training targets
train_pred = model.predict(X)
mse = np.mean(abs((train_pred - Y)))
print(f"Training MSE: {mse:.4f}")


Training MSE: 0.1032


In [23]:
# Persist artifacts and provide a small helper for inference
# output_dir = Path("cell-eval-outdir/baseline_ridge")
# output_dir.mkdir(parents=True, exist_ok=True)

# np.savez_compressed(
#     output_dir / "ridge_baseline.npz",
#     target_genes=np.array(target_genes),
#     mu_ctrl=mu_ctrl,
#     coef=model.coef_,
#     intercept=model.intercept_,
#     alpha=alpha,
# )

# def predict_for_target(gene_name: str):
#     try:
#         idx = target_genes.index(gene_name)
#     except ValueError as exc:
#         raise KeyError(f"{gene_name} not in training targets") from exc
#     x = np.zeros((1, len(target_genes)), dtype=float)
#     x[0, idx] = 1.0
#     # model predicts the shift from control mean; add mu_ctrl for absolute expression
#     return mu_ctrl + model.predict(x)[0]

# example_gene = target_genes[0]
# example_signature = predict_for_target(example_gene)
# print(f"Example prediction for {example_gene}: shape {example_signature.shape}")
# print(f"Saved model artifacts to {output_dir}")


In [ ]:
#Evaluate differential expression (DE) and perturbation discrimination score (PDS)
true_sigs = Y  # ground-truth shifts from control for each target
pred_sigs = model.predict(np.eye(len(target_genes)))  # predicted shifts

def pearsonr_np(a, b):
    a = np.asarray(a) - np.asarray(a).mean()
    b = np.asarray(b) - np.asarray(b).mean()
    denom = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())
    return 0.0 if denom == 0 else float((a * b).sum() / denom)

per_target_r = [pearsonr_np(t, p) for t, p in zip(true_sigs, pred_sigs)]
mean_de = float(np.mean(per_target_r))

from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(pred_sigs, true_sigs)  #rows: predicted target, cols: true target
ranks = []
for i in range(sim.shape[0]):
    order = np.argsort(-sim[i])
    rank = int(np.where(order == i)[0][0]) + 1  #1 = perfect top-1 match
    ranks.append(rank)

pds = 1 - (np.mean(np.array(ranks) - 1) / (len(target_genes) - 1))

print(f"Mean DE score (Pearson r): {mean_de:.4f}")
print(f"Perturbation discrimination score: {pds:.4f}")


Mean DE score (Pearson r): 0.7662
First 5 per-target DE scores: [0.9479896926371416, 0.8378109898620241, 0.7836606250693426, -0.08017635565326144, 0.9065111182350405]
Perturbation discrimination score: 0.8928
